# NB05 — Evaluation, statistics and paper figures

**Project:** CardioMamba-Net · **Stage:** 5 of 5
`01_verify` → `02_preprocess` → `03_baselines` → `04_cardiomamba_train` → **`05_evaluate`**

---

## What this produces

Everything the manuscript needs, generated from the per-window metrics that NB03 and NB04 already
wrote — so **no model is retrained here**. Only the robustness section (§7) runs inference, and it
is optional.

| Output | Corresponds to |
|---|---|
| `table2_per_scenario.csv` | Their Table 2 — per-scenario, all models |
| `table3_rva_combined.csv` | Their Table 3 — the headline comparison |
| `table4_peak_detection.csv` | Their Table 4 — R-peak accuracy/precision/recall/F1 |
| `table5_hrv.csv` | Their Table 5 — μRR, σRR, μHR, σHR, RMSSD, **in real milliseconds** |
| `table6_ablation.csv` | Ours — the ablation ladder |
| `table7_significance.csv` | Ours — Wilcoxon signed-rank with Holm correction |
| `table8_budget.csv` | Ours — parameters, and correlation per million parameters |
| 10 figures | Bland–Altman ×2, per-subject box plots, qualitative grid, ablation, robustness, budget scatter |

## What the statistics are for

The baseline reports means and standard deviations and stops. That is not enough to claim a win.
Here every model pair is compared with a **Wilcoxon signed-rank test across folds**, and the
p-values are **Holm-corrected** for multiple comparisons — because with 10 ablation rungs there are
45 pairwise tests and roughly two of them will look significant by chance alone.

Bland–Altman with limits of agreement is the standard way to report agreement between two
measurement methods in clinical work, and it is what a reviewer from a medical journal will look
for on heart rate and HRV. A correlation coefficient alone does not tell them whether the method
can be trusted on an individual patient.

## ⚠️ Accelerator: **GPU T4 × 2** *(only needed for §7)*

Sections 1–6 and 8 run fine on CPU. If you only want the tables and figures, set
`CFG["RUN_ROBUSTNESS"] = False` and use **Accelerator: None**.

---
# 1 · Configuration

In [ ]:
CFG = {
    "DATA_REPO":  "Shanmuk4622/cr-rvs-radar-ecg-processed",
    "MODEL_REPO": "Shanmuk4622/cardiomamba-net",
    "HF_PRIVATE": False,
    "RUN_ID":     "nb05_evaluation_v1",

    "WORK":    "/kaggle/working/nb05",
    "SCRATCH": "/kaggle/temp/nb05",
    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_REQ_HOUR": 120,

    "RUN_ROBUSTNESS": True,          # needs GPU + checkpoints; set False for tables only
    "SNR_DB": [12, 6, 3, 0, -3],
    "ROBUST_MODELS": ["L9_full", "multireslinknet"],
    "ROBUST_MAX_WINDOWS": 800,

    "HEADLINE_EXP": "B_rva",
    "ALPHA": 0.05,
    "SEED": 1337,
}
import json
print(json.dumps(CFG, indent=2))

In [ ]:
import os, sys, gc, json, math, time, warnings, subprocess, itertools
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    import importlib.util
    miss = [x for x in p if importlib.util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=False)
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd
WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "tables", WORK / "figures"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))
pd.set_option("display.width", 240, "display.max_columns", 60)
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
MODULES = {
 "crvs_sync.py":    r"""
# crvs_sync.py -- resumable, rate-limited, interrupt-safe Hugging Face folder sync.
# Identical across NB01-NB05 so the cadence rules are enforced in exactly one place.
#   * push at most once per PUSH_INTERVAL_S (default 30 min)
#   * push immediately when a stage finishes            -> sync.stage_done("name")
#   * push immediately when execution is stopped        -> SIGINT / SIGTERM / atexit
#   * one upload_folder call per flush, behind a token bucket, backing off on 429
#   * resume by pulling the run folder back on startup
import os, json, time, random, threading, atexit, signal
from pathlib import Path
from datetime import datetime, timezone

class TokenBucket:
    # capacity = requests per hour, refilled continuously
    def __init__(self, per_hour=120):
        self.capacity = float(per_hour); self.tokens = float(per_hour)
        self.rate = per_hour / 3600.0; self.t = time.monotonic()
        self.lock = threading.Lock()
    def take(self, n=1, block=True, timeout=1200):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.rate)
                self.t = now
                if self.tokens >= n:
                    self.tokens -= n; return True
                need = (n - self.tokens) / self.rate
            if not block or time.monotonic() + need > deadline:
                return False
            time.sleep(min(need, 5.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_req_hour=120, retry_max=6,
                 verbose=True):
        from huggingface_hub import HfApi
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = push_interval_s
        self.bucket = TokenBucket(max_req_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = 0.0
        self._flag = threading.Event(); self._stop = threading.Event()
        self._lock = threading.Lock()
        self._pushes = 0; self._failures = 0
        self.history = self.local / "history.jsonl"
        self.state_path = self.local / "state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)
        if not self.private:
            try:
                self.api.update_repo_visibility(self.repo_id, private=False,
                                                repo_type=self.repo_type, token=self.token)
            except Exception:
                pass

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def log(self, event, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with open(self.history, "a") as f:
                f.write(json.dumps(rec, default=str) + "\n")
        except Exception:
            pass
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def save_state(self, state):
        tmp = self.state_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(state, indent=2, default=str)); tmp.replace(self.state_path)

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text())
            except Exception:
                pass
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        from huggingface_hub import snapshot_download
        try:
            self.bucket.take(1)
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(into or self.local),
                                  allow_patterns=allow_patterns)
            self.log("resume_pull_ok", path=str(p)); return True
        except Exception as e:
            self.log("resume_pull_empty", err=type(e).__name__); return False

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw); self._flag.set()

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.bucket.take(1, block=True, timeout=1800):
                self.log("rate_limited_giveup"); return False
            try:
                upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                              repo_type=self.repo_type, token=self.token,
                              commit_message=msg,
                              ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                               "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                self.log("push_ok", n=self._pushes, msg=msg); return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {e}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None):
        with self._lock:
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            m = msg or ((self.run_id + " final") if final else (self.run_id + " @ " + stamp + "Z"))
            ok = self._do_upload(m); self._flag.clear(); return ok

    def _loop(self):
        while not self._stop.is_set():
            self._stop.wait(20)
            if self._stop.is_set():
                break
            due = (time.time() - self._last_push) >= self.interval
            want = self._flag.is_set()
            if due or want:
                try:
                    tag = "stage" if want else "periodic"
                    self.flush(msg=self.run_id + " " + tag + " @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=str(e))

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, msg=self.run_id + " interrupted (sig " + str(signum) + ")")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._stop.is_set():
            return
        self.log("closing"); self._stop.set()
        try:
            self.flush(final=True)
        except Exception:
            pass
""",
 "crvs_data.py":    r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 3
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.rng = np.random.RandomState(seed)
        self._cache = {}

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            if self.rng.rand() < 0.5:
                x = x + self.rng.randn(*x.shape).astype(np.float32) * 0.01
            if self.rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * self.rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
""",
 "crvs_metrics.py": r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
""",
 "crvs_models.py":  r"""
# crvs_models.py -- the four baseline 1-D segmentation networks.
# All four are standardised the way Chowdhury et al. 2024 describe (section 3.1):
# 5 levels, 64 filters in the first level, doubling thereafter. Input (B, C_in, 1024).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def cbr(i, o, k=3, s=1):
    return nn.Sequential(nn.Conv1d(i, o, k, s, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(cbr(i, o), cbr(o, o))
    def forward(self, x):
        return self.b(x)

# ------------------------------------------------------------------ UNet
class UNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList()
        for i in range(levels - 1):
            self.downs.append(DoubleConv(chs[i], chs[i + 1]))
        self.bott = DoubleConv(chs[-1], chs[-1] * 2)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        prev = chs[-1] * 2
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose1d(prev, c, 4, 2, 1))
            self.decs.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv1d(chs[0], out_ch, 1)
    def forward(self, x):
        skips = []
        h = self.inc(x); skips.append(h)
        for d in self.downs:
            h = d(F.max_pool1d(h, 2)); skips.append(h)
        h = self.bott(F.max_pool1d(h, 2))
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            h = up(h)
            if h.shape[-1] != sk.shape[-1]:
                h = F.interpolate(h, size=sk.shape[-1], mode="linear", align_corners=False)
            h = dec(torch.cat([h, sk], 1))
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ LinkNet
class LinkEnc(nn.Module):
    def __init__(self, i, o, stride=2):
        super().__init__()
        self.c1 = cbr(i, o, 3, stride)
        self.c2 = nn.Sequential(nn.Conv1d(o, o, 3, 1, 1, bias=False), nn.BatchNorm1d(o))
        self.sc = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))
    def forward(self, x):
        return F.relu(self.c2(self.c1(x)) + self.sc(x))

class LinkDec(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        m = max(i // 4, 8)
        self.a = cbr(i, m, 1)
        self.b = nn.Sequential(nn.ConvTranspose1d(m, m, 4, 2, 1, bias=False),
                               nn.BatchNorm1d(m), nn.ReLU(inplace=True))
        self.c = cbr(m, o, 1)
    def forward(self, x):
        return self.c(self.b(self.a(x)))

class LinkNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.bott = cbr(prev, prev)
        self.decs = nn.ModuleList()
        rev = list(reversed(chs))
        for k, c in enumerate(rev):
            nxt = rev[k + 1] if k + 1 < len(rev) else chs[0]
            self.decs.append(LinkDec(c, nxt))
        self.head = nn.Sequential(cbr(chs[0], chs[0]), nn.Conv1d(chs[0], out_ch, 1))
    def forward(self, x):
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 2 - k
            if j >= 0:
                s = skips[j]
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                h = h + s
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ FPN
class FPN1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, pyr=128):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.lat = nn.ModuleList([nn.Conv1d(c, pyr, 1) for c in chs])
        self.smooth = nn.ModuleList([cbr(pyr, pyr) for _ in chs])
        self.heads = nn.ModuleList([nn.Sequential(cbr(pyr, pyr // 2), cbr(pyr // 2, pyr // 2))
                                    for _ in chs])
        self.head = nn.Sequential(cbr(pyr // 2, pyr // 2), nn.Conv1d(pyr // 2, out_ch, 1))
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x); feats = []
        for e in self.encs:
            h = e(h); feats.append(h)
        ps = [None] * len(feats)
        ps[-1] = self.lat[-1](feats[-1])
        for i in range(len(feats) - 2, -1, -1):
            up = F.interpolate(ps[i + 1], size=feats[i].shape[-1], mode="linear",
                               align_corners=False)
            ps[i] = self.lat[i](feats[i]) + up
        ps = [s(p) for s, p in zip(self.smooth, ps)]
        acc = None
        for hd, p in zip(self.heads, ps):
            v = F.interpolate(hd(p), size=L, mode="linear", align_corners=False)
            acc = v if acc is None else acc + v
        return {"wave": torch.tanh(self.head(acc))}

# ------------------------------------------------------------------ MultiResLinkNet
class MultiResBlock(nn.Module):
    # MultiResUNet block (Ibtehaz & Rahman) in 1-D: three successive 3-conv stages of
    # increasing width, concatenated, plus a 1x1 residual shortcut.
    def __init__(self, cin, U, alpha=1.67):
        super().__init__()
        W = alpha * U
        # max(1, ...): below U=4 the 0.167 stage floors to zero channels, and the failure
        # then surfaces as an opaque conv error rather than pointing here.
        c1, c2, c3 = (max(1, int(W * 0.167)), max(1, int(W * 0.333)), max(1, int(W * 0.5)))
        self.out_channels = c1 + c2 + c3
        self.sc = nn.Sequential(nn.Conv1d(cin, self.out_channels, 1, bias=False),
                                nn.BatchNorm1d(self.out_channels))
        self.a = cbr(cin, c1); self.b = cbr(c1, c2); self.c = cbr(c2, c3)
        self.bn1 = nn.BatchNorm1d(self.out_channels)
        self.bn2 = nn.BatchNorm1d(self.out_channels)
    def forward(self, x):
        s = self.sc(x)
        a = self.a(x); b = self.b(a); c = self.c(b)
        o = self.bn1(torch.cat([a, b, c], 1))
        return F.relu(self.bn2(o + s))

class ResPath(nn.Module):
    # Processes an encoder feature before it is added to the decoder, instead of a raw skip.
    def __init__(self, ch, length):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(max(1, length)):
            self.blocks.append(nn.ModuleDict({
                "sc": nn.Sequential(nn.Conv1d(ch, ch, 1, bias=False), nn.BatchNorm1d(ch)),
                "cv": nn.Sequential(nn.Conv1d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm1d(ch)),
            }))
    def forward(self, x):
        for b in self.blocks:
            x = F.relu(b["sc"](x) + b["cv"](x))
        return x

class MultiResLinkNet1D(nn.Module):
    # LinkNet skeleton, MultiRes blocks instead of plain convolutions, ResPath skips added
    # (not concatenated), and deep supervision from every encoder level.
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        units = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        self.bott = MultiResBlock(prev, units[-1])
        self.decs = nn.ModuleList()
        rev_ch = list(reversed(enc_ch))
        cur = self.bott.out_channels
        # Decoder step k must emerge with the channel count AND length of skips[-1-k], or
        # the additive skip is silently dropped and every ResPath receives zero gradient.
        # Encoder here pools AFTER appending the skip, so the target is rev_ch[k] -- not
        # rev_ch[k+1], which is correct only for the stride-2 encoder in LinkNet1D.
        for k in range(levels):
            tgt = rev_ch[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.head = nn.Sequential(cbr(cur, base), nn.Conv1d(base, out_ch, 1))
        self.aux = nn.ModuleList([nn.Conv1d(c, out_ch, 1) for c in enc_ch]) \
                   if deep_supervision else None
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(                     # raise, not assert: an invariant
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs "        # this
                        f"{s.shape[1]}. Dropping it silently is what cost 59% of this "  # load
                        "model's gradient once already.")   # bearing must survive python -O
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {"wave": torch.tanh(self.head(h))}
        if self.aux is not None and self.training:
            out["aux"] = [F.interpolate(a(s), size=L, mode="linear", align_corners=False)
                          for a, s in zip(self.aux, skips)]
        return out

BASELINES = {"fpn": FPN1D, "unet": UNet1D, "linknet": LinkNet1D,
             "multireslinknet": MultiResLinkNet1D}

def build_baseline(name, in_ch=1, out_ch=1, base=64, levels=4):
    return BASELINES[name](in_ch=in_ch, out_ch=out_ch, base=base, levels=levels)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
""",
 "crvs_cmnet.py":   r"""
# crvs_cmnet.py -- CardioMamba-Net (contributions C1-C5 of PLAN.md).
# C2 dual-domain encoder, C3 bidirectional SSM bottleneck, C4 multi-task decoder with
# peak-conditioned FiLM refinement. C1 lives in the data pipeline, C5 in crvs_losses.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from crvs_models import cbr, MultiResBlock, ResPath, LinkDec, count_params

class ECA(nn.Module):
    # Efficient channel attention: a length-k 1-D conv over the channel descriptor.
    def __init__(self, ch, k=5):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        w = x.mean(-1, keepdim=True).transpose(1, 2)
        w = torch.sigmoid(self.conv(w)).transpose(1, 2)
        return x * w

class LiftingUnit(nn.Module):
    # Learnable second-generation wavelet: split into even/odd, predict, update.
    # Replaces a fixed wavelet basis with one the network chooses for radar.
    def __init__(self, ch, k=5):
        super().__init__()
        self.P = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
        self.U = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
    def forward(self, x):
        xe, xo = x[..., ::2], x[..., 1::2]
        n = min(xe.shape[-1], xo.shape[-1])
        xe, xo = xe[..., :n], xo[..., :n]
        d = xo - self.P(xe)
        c = xe + self.U(d)
        return c, d

class WaveletBranch(nn.Module):
    # Multi-resolution analysis producing one feature map per scale, to sit alongside the
    # convolutional branch. LifWavNet uses this idea as the whole network; here it is half
    # of a dual-domain encoder.
    #
    # Level 0 is taken at the INPUT resolution, before any lifting. Encoder level i sits at
    # L/2^i, and a lifting unit halves length, so starting the branch with a lifting step
    # would put every wavelet feature one octave below its conv counterpart and force a 2x
    # upsample at every fusion. The dual-domain claim (C2) is that the two branches see the
    # SAME scale from different domains, so they have to be aligned octave for octave.
    def __init__(self, ch, out_chs, levels=4):
        super().__init__()
        self.proj0 = nn.Conv1d(ch, out_chs[0], 1)
        self.units = nn.ModuleList([LiftingUnit(ch) for _ in range(max(levels - 1, 0))])
        self.proj = nn.ModuleList([nn.Conv1d(ch * 2, o, 1) for o in out_chs[1:]])
    def forward(self, x):
        feats = [self.proj0(x)]
        c = x
        for u, p in zip(self.units, self.proj):
            c, d = u(c)
            feats.append(p(torch.cat([c, d], 1)))
        return feats

class S4D(nn.Module):
    # Diagonal state-space layer (S4D-Lin). Pure PyTorch: an FFT convolution with a kernel
    # built from learned diagonal dynamics. No custom CUDA, so it always builds on Kaggle.
    def __init__(self, d_model, d_state=64, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        H, N = d_model, d_state // 2
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)
        self.log_A_real = nn.Parameter(torch.log(0.5 * torch.ones(H, N)))
        self.A_imag = nn.Parameter(math.pi * torch.arange(N).float().repeat(H, 1))
        self.C = nn.Parameter(torch.randn(H, N, 2) * (0.5 ** 0.5))
        self.D = nn.Parameter(torch.randn(H))
    def kernel(self, L, device, dtype=torch.float32):
        dt = torch.exp(self.log_dt).to(dtype).unsqueeze(-1)
        A = -torch.exp(self.log_A_real.to(dtype)) + 1j * self.A_imag.to(dtype)
        C = torch.view_as_complex(self.C.to(dtype).contiguous())
        dtA = A * dt
        n = torch.arange(L, device=device, dtype=dtype)
        K = dtA.unsqueeze(-1) * n
        Cc = C * (torch.exp(dtA) - 1.0) / A
        return 2.0 * torch.einsum("hn,hnl->hl", Cc, torch.exp(K)).real
    def forward(self, u):
        L = u.shape[-1]
        uf = u.float()
        k = self.kernel(L, u.device)
        n = 2 * L
        y = torch.fft.irfft(torch.fft.rfft(uf, n=n) * torch.fft.rfft(k, n=n), n=n)[..., :L]
        y = y + uf * self.D.unsqueeze(-1)
        return y.to(u.dtype)

class BiSSM(nn.Module):
    # Bidirectional SSM block: an 8 s window holds 8-10 cardiac cycles, and this is what
    # lets beat n inform beat n+1. Linear time in sequence length.
    def __init__(self, d, d_state=64, expand=2, dropout=0.1):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.fwd = S4D(d, d_state)
        self.bwd = S4D(d, d_state)
        self.mix = nn.Conv1d(2 * d, d, 1)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Conv1d(d, expand * d, 1), nn.GELU(),
                                nn.Dropout(dropout), nn.Conv1d(expand * d, d, 1))
    def forward(self, x):
        h = self.n1(x.transpose(1, 2)).transpose(1, 2)
        f = self.fwd(h)
        b = self.bwd(h.flip(-1)).flip(-1)
        x = x + self.mix(torch.cat([f, b], 1))
        h = self.n2(x.transpose(1, 2)).transpose(1, 2)
        return x + self.ff(h)

class TransformerBottleneck(nn.Module):
    # The fair-fight control for C3: same budget, attention instead of an SSM.
    def __init__(self, d, nhead=8, layers=3, dropout=0.1, max_len=1024):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        lyr = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=2 * d, dropout=dropout,
                                         batch_first=True, norm_first=True,
                                         activation="gelu")
        self.enc = nn.TransformerEncoder(lyr, layers)
    def forward(self, x):
        h = x.transpose(1, 2)
        h = h + self.pos[:, :h.shape[1]]
        return self.enc(h).transpose(1, 2)

class FiLM(nn.Module):
    # Peak-conditioned refinement: the R-peak head tells the waveform head where a QRS
    # belongs BEFORE it draws one. Modulation is per-sample, because a QRS is localised.
    def __init__(self, cond_ch, feat_ch, k=9):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(cond_ch, feat_ch, k, padding=k // 2), nn.GELU(),
                                 nn.Conv1d(feat_ch, 2 * feat_ch, 1))
    def forward(self, feat, cond):
        g, b = self.net(cond).chunk(2, 1)
        return feat * (1.0 + torch.tanh(g)) + b

class CardioMambaNet(nn.Module):
    def __init__(self, in_ch=8, base=32, levels=4, d_ssm=256, ssm_blocks=3, d_state=64,
                 bottleneck="ssm", use_wavelet=True, multitask=True, use_film=True,
                 dropout=0.1):
        super().__init__()
        self.use_wavelet = use_wavelet
        self.multitask = multitask
        self.use_film = use_film and multitask
        units = [base * (2 ** i) for i in range(levels)]
        # C1 lands here: a learnable 1x1 mix over the 8 physics channels, so the network
        # can rediscover arctangent demodulation if that really is optimal.
        self.mix = nn.Sequential(nn.Conv1d(in_ch, 32, 1), nn.GELU())
        self.stem = cbr(32, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        if use_wavelet:
            self.wave = WaveletBranch(base, enc_ch, levels)
            self.fuse = nn.ModuleList([nn.Sequential(nn.Conv1d(2 * c, c, 1), ECA(c))
                                       for c in enc_ch])
        self.pre = nn.Conv1d(prev, d_ssm, 1)
        if bottleneck == "ssm":
            self.bott = nn.Sequential(*[BiSSM(d_ssm, d_state, dropout=dropout)
                                        for _ in range(ssm_blocks)])
        elif bottleneck == "transformer":
            self.bott = TransformerBottleneck(d_ssm, layers=ssm_blocks, dropout=dropout)
        else:
            self.bott = nn.Sequential(cbr(d_ssm, d_ssm), cbr(d_ssm, d_ssm))
        self.post = nn.Conv1d(d_ssm, prev, 1)
        self.decs = nn.ModuleList()
        rev = list(reversed(enc_ch)); cur = prev
        # Same indexing rule as MultiResLinkNet1D: target rev[k], so decoder step k lines up
        # with skips[-1-k] in both channels and length and the ResPath actually contributes.
        for k in range(levels):
            tgt = rev[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.refine = cbr(cur, base)
        self.head_wave = nn.Conv1d(base, 1, 1)
        if multitask:
            self.head_peak = nn.Sequential(cbr(cur, base), nn.Conv1d(base, 1, 1))
            rr_ch = max(base // 2, 4)
            self.head_rr = nn.Sequential(cbr(cur, rr_ch), nn.Conv1d(rr_ch, 1, 1))
            if self.use_film:
                self.film = FiLM(1, base)
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(self.mix(x))
        wfeat = self.wave(h) if self.use_wavelet else None
        skips = []
        for i, e in enumerate(self.encs):
            h = e(h)
            if wfeat is not None:
                w = wfeat[i]
                if w.shape[-1] != h.shape[-1]:
                    w = F.interpolate(w, size=h.shape[-1], mode="linear", align_corners=False)
                h = self.fuse[i](torch.cat([h, w], 1))
            skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.post(self.bott(self.pre(h)))
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs {s.shape[1]}")
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {}
        if self.multitask:
            peak_logit = self.head_peak(h)
            out["peak"] = peak_logit
            out["rr"] = F.softplus(self.head_rr(h))
            f = self.refine(h)
            if self.use_film:
                # Gradient flows through the conditioning on purpose: the peak head is meant
                # to be shaped by the waveform loss as well as its own, which is the point of
                # peak-conditioned refinement. Detaching here would make it a one-way hint.
                f = self.film(f, torch.sigmoid(peak_logit))
            out["wave"] = torch.tanh(self.head_wave(f))
        else:
            out["wave"] = torch.tanh(self.head_wave(self.refine(h)))
        return out

def build_cmnet(**kw):
    return CardioMambaNet(**kw)
""",
 "crvs_losses.py":  r"""
# crvs_losses.py -- C5, the morphology-aware composite loss.
# Plain MSE is the conditional mean, so it flattens the R peak; that is exactly why the
# baseline over-estimates RMSSD by ~2x. Every term here exists to stop that.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiResSTFTLoss(nn.Module):
    # Spectral convergence + log-magnitude at three resolutions. Forces the model to get
    # the spectrum right, not just the sample-wise average.
    def __init__(self, ffts=(256, 128, 64)):
        super().__init__()
        self.ffts = ffts
    def _one(self, y, yh, n):
        hop, win = n // 4, n
        w = torch.hann_window(win, device=y.device, dtype=torch.float32)
        kw = dict(n_fft=n, hop_length=hop, win_length=win, window=w,
                  return_complex=True, center=True, pad_mode="reflect")
        Y = torch.stft(y, **kw).abs().clamp_min(1e-7)
        H = torch.stft(yh, **kw).abs().clamp_min(1e-7)
        sc = torch.norm(Y - H, p="fro", dim=(-2, -1)) / (torch.norm(Y, p="fro", dim=(-2, -1)) + 1e-7)
        mag = F.l1_loss(torch.log(H), torch.log(Y))
        return sc.mean() + mag
    def forward(self, y, yh):
        y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
        return sum(self._one(y, yh, n) for n in self.ffts) / len(self.ffts)

def pearson_loss(y, yh, eps=1e-8):
    y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
    y = y - y.mean(-1, keepdim=True); yh = yh - yh.mean(-1, keepdim=True)
    num = (y * yh).sum(-1)
    den = y.norm(dim=-1) * yh.norm(dim=-1) + eps
    return (1.0 - num / den).mean()

def focal_bce(logit, target, alpha=0.75, gamma=2.0):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pt = p * target + (1 - p) * (1 - target)
    w = alpha * target + (1 - alpha) * (1 - target)
    return (w * (1 - pt).pow(gamma) * ce).mean()

class CompositeLoss(nn.Module):
    def __init__(self, w_huber=1.0, w_stft=0.5, w_peak=0.3, w_rr=0.1,
                 w_peakw=0.5, w_corr=0.3, huber_delta=0.1, peak_weight=4.0):
        super().__init__()
        self.w = dict(huber=w_huber, stft=w_stft, peak=w_peak, rr=w_rr,
                      peakw=w_peakw, corr=w_corr)
        self.delta = huber_delta
        self.peak_weight = peak_weight
        self.stft = MultiResSTFTLoss()
    def forward(self, pred, y, pk=None, rr=None):
        parts = {}
        wave = pred["wave"]
        if self.w["huber"]:
            parts["huber"] = F.huber_loss(wave, y, delta=self.delta)
        if self.w["stft"]:
            parts["stft"] = self.stft(y, wave)
        if self.w["corr"]:
            parts["corr"] = pearson_loss(y, wave)
        if self.w["peakw"] and pk is not None:
            wgt = 1.0 + self.peak_weight * pk
            parts["peakw"] = ((wgt * (wave - y).abs()).sum() / (wgt.sum() + 1e-8))
        if self.w["peak"] and pk is not None and "peak" in pred:
            parts["peak"] = focal_bce(pred["peak"], pk)
        if self.w["rr"] and rr is not None and "rr" in pred:
            parts["rr"] = F.l1_loss(pred["rr"], rr)
        if "aux" in pred:
            parts["aux"] = sum(F.huber_loss(a, y, delta=self.delta)
                               for a in pred["aux"]) / max(len(pred["aux"]), 1) * 0.2
        total = sum(self.w.get(k, 1.0) * v for k, v in parts.items())
        return total, {k: float(v.detach()) for k, v in parts.items()}

class MSEOnly(nn.Module):
    # The baseline's objective, kept verbatim so ablation row 1 is a true reproduction.
    def forward(self, pred, y, pk=None, rr=None):
        l = F.mse_loss(pred["wave"], y)
        if "aux" in pred:
            l = l + 0.2 * sum(F.mse_loss(a, y) for a in pred["aux"]) / max(len(pred["aux"]), 1)
        return l, {"mse": float(l.detach())}
""",
 "crvs_engine.py":  r"""
# crvs_engine.py -- the shared GPU training engine.
# Dual T4 via DataParallel, AMP, cosine schedule, early stopping, and a checkpoint written
# EVERY epoch so a killed session costs nothing. Runs are queued and skipped if already done.
import json, math, time, os
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def _autocast(device_type, enabled):
    # torch.cuda.amp.autocast / GradScaler are deprecated and warn on every step in
    # torch >= 2.4. Use the device-typed API where it exists, fall back where it does not.
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def _grad_scaler(device_type, enabled):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def pick_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        return torch.device("cuda"), n, names
    return torch.device("cpu"), 0, []

def seed_all(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

class Trainer:
    def __init__(self, model, loss_fn, out_dir, run_id, sync=None, lr=5e-4, weight_decay=1e-4,
                 epochs=120, patience=20, batch_size=64, num_workers=2, amp=True,
                 multi_gpu=True, grad_clip=1.0, min_lr=1e-6, log_every=50):
        self.device, self.ngpu, self.gpu_names = pick_device()
        self.raw_model = model.to(self.device)
        self.model = self.raw_model
        if multi_gpu and self.ngpu > 1:
            self.model = nn.DataParallel(self.raw_model)
        self.loss_fn = loss_fn
        self.out = Path(out_dir); self.out.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id; self.sync = sync
        self.epochs = epochs; self.patience = patience
        self.bs = batch_size; self.nw = num_workers
        self.amp = amp and self.device.type == "cuda"
        self.grad_clip = grad_clip
        self.opt = torch.optim.AdamW(self.raw_model.parameters(), lr=lr,
                                     weight_decay=weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=epochs,
                                                                eta_min=min_lr)
        self.scaler = _grad_scaler(self.device.type, self.amp)
        self.log_every = log_every
        self.state = {"epoch": 0, "best": float("inf"), "best_epoch": -1, "history": [],
                      "run_id": run_id, "done": False}

    @property
    def ckpt(self):
        return self.out / "state.pt"

    def save(self, tag="state"):
        torch.save({"model": self.raw_model.state_dict(),
                    "opt": self.opt.state_dict(),
                    "sched": self.sched.state_dict(),
                    "scaler": self.scaler.state_dict(),
                    "state": self.state,
                    "torch_rng": torch.get_rng_state(),
                    "np_rng": np.random.get_state()},
                   self.out / (tag + ".pt"))
        (self.out / "state.json").write_text(json.dumps(self.state, indent=2, default=str))

    def load(self):
        if not self.ckpt.exists():
            return False
        try:
            d = torch.load(self.ckpt, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(d["model"])
            self.opt.load_state_dict(d["opt"]); self.sched.load_state_dict(d["sched"])
            self.scaler.load_state_dict(d["scaler"]); self.state = d["state"]
            try:
                torch.set_rng_state(d["torch_rng"].cpu()); np.random.set_state(d["np_rng"])
            except Exception:
                pass
            print(f"  resumed {self.run_id} at epoch {self.state['epoch']}")
            return True
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__}), starting fresh")
            return False

    def _loader(self, ds, shuffle):
        if len(ds) == 0:
            raise RuntimeError("empty dataset -- check the fold split; training on nothing "
                               "would produce a plausible-looking but untrained checkpoint")
        # drop_last=True on a split smaller than one batch yields ZERO batches, the optimiser
        # never steps, and the loss is reported as 0.00000. Guard it explicitly.
        drop = bool(shuffle) and len(ds) > self.bs
        if shuffle and not drop:
            print(f"  note: only {len(ds)} train window(s) < batch {self.bs}; keeping the "
                  f"partial batch so the optimiser actually steps")
        return DataLoader(ds, batch_size=min(self.bs, max(len(ds), 1)), shuffle=shuffle,
                          num_workers=self.nw, pin_memory=(self.device.type == "cuda"),
                          drop_last=drop, persistent_workers=self.nw > 0)

    def _step(self, batch, train):
        x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
        with _autocast(self.device.type, self.amp):
            pred = self.model(x)
            if isinstance(pred, dict) and "aux" in pred and not train:
                pred = {k: v for k, v in pred.items() if k != "aux"}
            loss, parts = self.loss_fn(pred, y, pk, rr)
        return loss, parts, pred, y

    def fit(self, train_ds, val_ds):
        tl = self._loader(train_ds, True)
        vl = self._loader(val_ds, False)
        start = self.state["epoch"]
        # Only the epoch counter decides completion. Keying off a sticky `done` flag meant
        # raising CFG["EPOCHS"] later silently no-opped instead of training further.
        if start >= self.epochs:
            print(f"  {self.run_id} already complete at epoch {start}/{self.epochs}")
            return self.state
        if self.state.get("done"):
            print(f"  extending {self.run_id}: {start} -> {self.epochs} epochs")
            self.state["done"] = False
        bad = 0
        for ep in range(start, self.epochs):
            self.model.train(); t0 = time.time(); tot = 0.0; n = 0
            for i, batch in enumerate(tl):
                self.opt.zero_grad(set_to_none=True)
                loss, parts, _, _ = self._step(batch, True)
                self.scaler.scale(loss).backward()
                if self.grad_clip:
                    self.scaler.unscale_(self.opt)
                    torch.nn.utils.clip_grad_norm_(self.raw_model.parameters(), self.grad_clip)
                self.scaler.step(self.opt); self.scaler.update()
                tot += float(loss.detach()); n += 1
            if n == 0:
                raise RuntimeError(
                    "the training loader yielded zero batches -- the optimiser never stepped. "
                    "This would write a checkpoint that looks trained and is not.")
            self.sched.step()
            tr = tot / n
            self.model.eval(); vtot = 0.0; vn = 0
            with torch.no_grad():
                for batch in vl:
                    loss, _, _, _ = self._step(batch, False)
                    vtot += float(loss); vn += 1
            va = vtot / max(vn, 1)
            rec = {"epoch": ep + 1, "train": tr, "val": va,
                   "lr": self.opt.param_groups[0]["lr"], "sec": round(time.time() - t0, 1)}
            self.state["history"].append(rec); self.state["epoch"] = ep + 1
            improved = va < self.state["best"] - 1e-6
            if improved:
                self.state["best"] = va; self.state["best_epoch"] = ep + 1; bad = 0
                self.save("best")
            else:
                bad += 1
            self.save("state")
            if self.sync:
                self.sync.log("epoch", run=self.run_id, **rec, best=round(self.state["best"], 6))
                if improved:
                    self.sync.stage_done(f"{self.run_id}:best@{ep+1}")
            print(f"  ep {ep+1:>3}/{self.epochs}  train {tr:.5f}  val {va:.5f}"
                  f"{'  *' if improved else ''}  {rec['sec']:.0f}s")
            if bad >= self.patience:
                print(f"  early stop at epoch {ep+1} (no improvement for {self.patience})")
                break
        self.state["done"] = True; self.save("state")
        if self.sync:
            self.sync.stage_done(f"{self.run_id}:done")
        return self.state

    @torch.no_grad()
    def predict(self, ds, max_keep=200):
        bp = self.out / "best.pt"
        if bp.exists():
            try:
                self.raw_model.load_state_dict(
                    torch.load(bp, map_location=self.device, weights_only=False)["model"])
            except Exception as e:
                print("  could not load best.pt:", e)
        self.model.eval()
        dl = self._loader(ds, False)
        Y, P = [], []
        for batch in dl:
            x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
            with _autocast(self.device.type, self.amp):
                out = self.model(x)
            Y.append(y.squeeze(1).float().cpu().numpy())
            P.append(out["wave"].squeeze(1).float().cpu().numpy())
        Y = np.concatenate(Y, 0); P = np.concatenate(P, 0)
        return Y, P
""",
}
for nm, src in MODULES.items():
    (WORK / nm).write_text(src)
import importlib
for nm in MODULES:
    sys.modules.pop(nm[:-3], None)
importlib.invalidate_caches()
import crvs_data
REQUIRED_LIB = 3
if getattr(crvs_data, "LIB_VERSION", 0) < REQUIRED_LIB:
    raise RuntimeError(f"stale crvs_data v{getattr(crvs_data,'LIB_VERSION','missing')}, "
                       f"need >= {REQUIRED_LIB}. Restart the kernel.")
print(f"library written ({len(MODULES)} modules), crvs_data v{crvs_data.LIB_VERSION}")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found. Add-ons -> Secrets -> HF_TOKEN (write) -> attach.\n" + "="*74)

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["MODEL_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="model",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"], max_req_hour=CFG["HF_MAX_REQ_HOUR"])
print("\nresults repo:", sync.url)
_M = {"f": False, "n": ""}
def MAJOR(nm):
    _M["f"] = True; _M["n"] = nm
def _hook(r=None):
    if _M["f"]:
        nm = _M["n"]; _M["f"] = False; _M["n"] = ""; sync.stage_done(nm)
try:
    get_ipython().events.register("post_run_cell", _hook)
except Exception:
    pass
MAJOR("00_setup")

---
# 2 · Collect every run

Pull the per-run summaries, per-window metrics and per-subject HRV from the model repo.
Checkpoints (`*.pt`) are **not** downloaded unless §7 runs — they are the bulk of the repo and the
tables do not need them.

In [ ]:
from huggingface_hub import snapshot_download
RUNS = SCRATCH / "runs"
pats = ["runs/**/summary.json", "runs/**/metrics_windows.parquet",
        "runs/**/metrics_subjects.parquet", "runs/**/preds_sample.npz",
        "runs/**/state.json", "results/*", "README.md"]
if CFG["RUN_ROBUSTNESS"]:
    pats.append("runs/**/best.pt")
t0 = time.time()
snapshot_download(CFG["MODEL_REPO"], repo_type="model", token=HF_TOKEN,
                  local_dir=str(RUNS), allow_patterns=pats)
print(f"downloaded in {time.time()-t0:.0f}s")

def norm_variant(s):
    v = s.get("variant") or s.get("model")
    if v == "multireslinknet" and s.get("loss", "mse") == "mse":
        return "multireslinknet"
    return v

rows, wrows, srows = [], [], []
for p in sorted((RUNS / "runs").glob("*/summary.json")):
    try:
        s = json.loads(p.read_text())
    except Exception:
        continue
    v = norm_variant(s)
    base = {"run_id": s["run_id"], "experiment": s["experiment"], "variant": v,
            "fold": s["fold"], "params": s.get("params"), "best_epoch": s.get("best_epoch")}
    rows.append({**base, **{k: val for k, val in s["metrics"].items() if not k.endswith("_std")}})
    mw = p.parent / "metrics_windows.parquet"
    if mw.exists():
        d = pd.read_parquet(mw); d["variant"] = v; d["experiment"] = s["experiment"]
        d["fold"] = s["fold"]; wrows.append(d)
    msj = p.parent / "metrics_subjects.parquet"
    if msj.exists():
        d = pd.read_parquet(msj); d["variant"] = v; d["experiment"] = s["experiment"]
        d["fold"] = s["fold"]; srows.append(d)

R  = pd.DataFrame(rows)
WD = pd.concat(wrows, ignore_index=True) if wrows else pd.DataFrame()
SD = pd.concat(srows, ignore_index=True) if srows else pd.DataFrame()
if not len(R):
    raise RuntimeError("No runs found. Run NB03 and NB04 first.")
R.to_csv(WORK / "tables" / "all_runs.csv", index=False)
print(f"runs: {len(R)}   window rows: {len(WD):,}   subject rows: {len(SD):,}")
print("\nruns per experiment x variant:")
print(R.pivot_table(index="variant", columns="experiment", values="fold",
                    aggfunc="count", fill_value=0).to_string())

---
# 3 · Tables 2 and 3 — the direct comparison

Same layout as the baseline's tables so a reader can put them side by side. The `_paper` columns
are their published figures; `Δ` is ours minus theirs.

A reminder on how to read the sign. Our splits are strictly subject-wise with non-overlapping test
windows, which theirs almost certainly were not. So a **baseline** row landing below its published
value is expected — that is leakage being removed. The claim we are making is about
**CardioMamba-Net versus our own re-run baselines**, under identical conditions. The published
column is context, not the yardstick.

In [ ]:
COLS  = ["MAE", "MSE", "CC_temporal", "CC_spectral", "RRMSE_temporal", "RRMSE_spectral"]
ORDER = ["fpn", "unet", "linknet", "multireslinknet",
         "L2_loss_only", "L3_c1_only", "L4_c1_c5", "L5_no_wavelet", "L6_no_ssm",
         "L7_singletask", "L8_no_film", "L9_full", "L10_transformer"]
NICE = {"fpn": "FPN", "unet": "UNet", "linknet": "LinkNet",
        "multireslinknet": "MultiResLinkNet", "L9_full": "CardioMamba-Net (ours)",
        "L10_transformer": "CardioMamba-Net (Transformer)"}

PAPER_A = {
 ("A_resting","fpn"):(0.14204,0.03170,58.37,71.38,0.46940,0.73374),
 ("A_resting","unet"):(0.13872,0.03219,63.10,74.68,0.45760,0.86096),
 ("A_resting","linknet"):(0.13588,0.03034,64.35,74.37,0.45116,0.81111),
 ("A_resting","multireslinknet"):(0.13258,0.03066,66.10,82.44,0.43682,0.71412),
 ("A_valsalva","fpn"):(0.14985,0.03679,57.53,65.97,0.46395,0.87990),
 ("A_valsalva","unet"):(0.15249,0.03928,58.38,68.79,0.46553,0.99554),
 ("A_valsalva","linknet"):(0.15087,0.03869,56.63,66.87,0.46195,0.99095),
 ("A_valsalva","multireslinknet"):(0.15286,0.04012,60.14,77.05,0.46083,0.80660),
 ("A_apnea","fpn"):(0.15310,0.03853,39.12,51.26,0.51017,1.00889),
 ("A_apnea","unet"):(0.14406,0.03495,56.14,69.97,0.47825,0.92034),
 ("A_apnea","linknet"):(0.14572,0.03526,56.22,70.35,0.47944,0.91749),
 ("A_apnea","multireslinknet"):(0.14474,0.03474,55.33,74.66,0.47692,0.82392),
 ("B_rva","fpn"):(0.14316,0.03422,59.63,69.53,0.44694,0.83026),
 ("B_rva","unet"):(0.14798,0.03741,57.65,68.39,0.45315,0.94118),
 ("B_rva","linknet"):(0.14780,0.03723,58.69,70.91,0.45487,0.86909),
 ("B_rva","multireslinknet"):(0.14841,0.03793,61.86,79.96,0.44618,0.73269),
}

def table_for(exps, fname, title):
    sub = R[R["experiment"].isin(exps)]
    if not len(sub):
        print(f"(no runs for {exps})"); return None
    g = sub.groupby(["experiment", "variant"])
    T = g[COLS].mean().round(5)
    Tsd = g[COLS].std().round(5)
    T["folds"] = g.size()
    T = T.reset_index()
    for i, r in T.iterrows():
        key = (r["experiment"], r["variant"])
        if key in PAPER_A:
            p = PAPER_A[key]
            T.loc[i, "CC_t_paper"] = p[2]; T.loc[i, "CC_s_paper"] = p[3]
            T.loc[i, "MAE_paper"] = p[0]
            T.loc[i, "dCC_t"] = round(r["CC_temporal"] - p[2], 2)
    T["order"] = T["variant"].map(lambda v: ORDER.index(v) if v in ORDER else 99)
    T = T.sort_values(["experiment", "order"]).drop(columns="order")
    T["variant"] = T["variant"].map(lambda v: NICE.get(v, v))
    print("=" * 130); print(title); print("=" * 130)
    print(T.to_string(index=False))
    T.to_csv(WORK / "tables" / fname, index=False)
    return T

T2 = table_for(["A_resting", "A_valsalva", "A_apnea"], "table2_per_scenario.csv",
               "TABLE 2  —  per scenario  (their Table 2)")
print()
T3 = table_for(["B_rva"], "table3_rva_combined.csv",
               "TABLE 3  —  Resting + Valsalva + Apnea combined  (their Table 3)")
print()
TC = table_for(["C_all5"], "table3b_all_five.csv",
               "TABLE 3b  —  ALL FIVE SCENARIOS  (new: the baseline never evaluated Tilt)")
MAJOR("01_tables_2_3")

---
# 4 · Tables 4 and 5 — beats and rhythm

Table 4 is R-peak detection on the reconstructed ECG. We add two columns the baseline does not
report: **median timing error in milliseconds** and **missed-detection rate**. Precision and recall
alone hide whether a "detected" beat is 10 ms or 90 ms off, and for HRV that difference is
everything.

Table 5 is the HRV comparison, and it is where the baseline's weakness is most visible. Their
predicted RMSSD is roughly **double** ground truth in every scenario (12.95 → 23.87 ms resting;
13.17 → 31.50 ms apnea) — the fingerprint of a smeared, jittery QRS. Our RMSSD error column is
the direct test of whether C5 fixed it.

We also report μRR in **genuine milliseconds**. Theirs is 126 ms alongside a heart rate of 62 bpm,
which is arithmetically impossible — 126 samples at 128 Hz is 0.98 s, so their column is samples
mislabelled as milliseconds.

In [ ]:
if len(SD):
    keep = [v for v in ORDER if v in set(SD["variant"])]
    g = SD[SD["experiment"] == CFG["HEADLINE_EXP"]].groupby("variant")
    T4 = g.agg(accuracy=("accuracy", "mean"), F1=("F1", "mean"),
               precision=("precision", "mean"), recall=("recall", "mean"),
               TP=("TP", "sum"), FP=("FP", "sum"), FN=("FN", "sum"),
               timing_err_ms=("timing_err_ms_median", "mean"),
               missed_rate=("missed_rate", "mean")).round(4)
    T4 = T4.reindex([v for v in keep if v in T4.index])
    T4.index = [NICE.get(i, i) for i in T4.index]
    print("=" * 118)
    print(f"TABLE 4  —  R-peak detection on the reconstructed ECG  ({CFG['HEADLINE_EXP']})")
    print("=" * 118)
    print(T4.to_string())
    print("\npublished (MultiResLinkNet, RVA): accuracy 0.886  F1 0.939  precision 0.973  recall 0.908")
    T4.to_csv(WORK / "tables" / "table4_peak_detection.csv")

    rows5 = []
    for v in keep:
        d = SD[(SD["variant"] == v) & (SD["experiment"] == CFG["HEADLINE_EXP"])]
        if not len(d):
            continue
        rows5.append({"variant": NICE.get(v, v), "signal": "ground truth",
                      "mu_RR_ms": d["gt_mean_rr_ms"].mean(), "sd_RR_ms": d["gt_sd_rr_ms"].mean(),
                      "mu_HR_bpm": d["gt_mean_hr_bpm"].mean(), "sd_HR_bpm": d["gt_sd_hr_bpm"].mean(),
                      "RMSSD_ms": d["gt_rmssd_ms"].mean()})
        rows5.append({"variant": NICE.get(v, v), "signal": "predicted",
                      "mu_RR_ms": d["pr_mean_rr_ms"].mean(), "sd_RR_ms": d["pr_sd_rr_ms"].mean(),
                      "mu_HR_bpm": d["pr_mean_hr_bpm"].mean(), "sd_HR_bpm": d["pr_sd_hr_bpm"].mean(),
                      "RMSSD_ms": d["pr_rmssd_ms"].mean()})
        rows5.append({"variant": NICE.get(v, v), "signal": "|error|",
                      "mu_RR_ms": (d["gt_mean_rr_ms"] - d["pr_mean_rr_ms"]).abs().mean(),
                      "sd_RR_ms": np.nan,
                      "mu_HR_bpm": (d["gt_mean_hr_bpm"] - d["pr_mean_hr_bpm"]).abs().mean(),
                      "sd_HR_bpm": np.nan,
                      "RMSSD_ms": (d["gt_rmssd_ms"] - d["pr_rmssd_ms"]).abs().mean()})
    T5 = pd.DataFrame(rows5).round(2)
    print("\n" + "=" * 118)
    print(f"TABLE 5  —  HR and HRV, in REAL milliseconds  ({CFG['HEADLINE_EXP']})")
    print("=" * 118)
    print(T5.to_string(index=False))
    T5.to_csv(WORK / "tables" / "table5_hrv.csv", index=False)
    print("\npublished (MultiResLinkNet, resting): RMSSD ground truth 12.95 ms -> predicted 23.87 ms")
    print("i.e. an 84 % over-estimate. The |error| rows above are the direct comparison.")
else:
    print("no per-subject metrics found -- NB03/NB04 write these; re-run them.")
MAJOR("02_tables_4_5")

---
# 5 · Table 6 — the ablation, and Table 7 — significance

Table 7 is what lets us write "significantly better" without a reviewer objecting. Wilcoxon
signed-rank is paired and non-parametric, which is right here: the same folds are used for every
model, and with five folds we have no business assuming normality.

Holm correction matters more than people expect. Ten rungs give 45 pairwise comparisons; at
α = 0.05 you would expect about two false positives. Holm controls the family-wise error rate
without the crushing conservatism of Bonferroni.

In [ ]:
from crvs_metrics import wilcoxon_holm

b = R[R["experiment"] == CFG["HEADLINE_EXP"]]
LAB6 = {"multireslinknet": "1. MultiResLinkNet + MSE (baseline)",
        "L2_loss_only": "2. + composite loss (C5)",
        "L3_c1_only": "3. + 8-channel input (C1)",
        "L4_c1_c5": "4. + C1 + C5",
        "L5_no_wavelet": "5. CardioMamba, no wavelet (-C2)",
        "L6_no_ssm": "6. CardioMamba, no SSM (-C3)",
        "L7_singletask": "7. CardioMamba, single-task (-C4)",
        "L8_no_film": "8. CardioMamba, no FiLM",
        "L9_full": "9. CardioMamba-Net (full)",
        "L10_transformer": "10. Transformer bottleneck (control)"}
lad = [v for v in LAB6 if v in set(b["variant"])]
if lad:
    extra = [c for c in ("peak_F1", "MAE_mean_hr_bpm", "MAE_rmssd_ms") if c in b.columns]
    T6 = b[b["variant"].isin(lad)].groupby("variant")[COLS + extra + ["params"]].mean()
    T6 = T6.reindex(lad)
    T6["folds"] = b.groupby("variant").size().reindex(lad)
    ref = T6.loc["multireslinknet", "CC_temporal"] if "multireslinknet" in T6.index else np.nan
    T6["dCC_t"] = (T6["CC_temporal"] - ref).round(2)
    T6.index = [LAB6[i] for i in T6.index]
    print("=" * 132); print("TABLE 6  —  ablation ladder"); print("=" * 132)
    print(T6.round(5).to_string())
    T6.to_csv(WORK / "tables" / "table6_ablation.csv")

    groups = {v: b[b["variant"] == v].sort_values("fold")["CC_temporal"].tolist() for v in lad}
    nf = min(len(x) for x in groups.values())
    if nf >= 3:
        res = wilcoxon_holm(groups)
        T7 = pd.DataFrame(res)
        T7["a"] = T7["a"].map(LAB6); T7["b"] = T7["b"].map(LAB6)
        T7["significant"] = T7["p_holm"] < CFG["ALPHA"]
        T7 = T7.sort_values("p_holm")
        print("\n" + "=" * 118)
        print(f"TABLE 7  —  Wilcoxon signed-rank on CC_temporal across {nf} folds, Holm-corrected")
        print("=" * 118)
        print(T7.to_string(index=False))
        T7.to_csv(WORK / "tables" / "table7_significance.csv", index=False)
        vs = T7[(T7["a"].str.contains("full")) | (T7["b"].str.contains("full"))]
        if len(vs):
            print(f"\n  comparisons involving the full model: "
                  f"{int(vs['significant'].sum())}/{len(vs)} significant at alpha={CFG['ALPHA']}")
    else:
        print(f"\nTABLE 7 skipped: only {nf} fold(s). Wilcoxon needs at least 3.")
        print("Run NB03/NB04 with QUICK=False so all 5 folds complete.")

if "params" in R.columns and R["params"].notna().any():
    T8 = (R[R["experiment"] == CFG["HEADLINE_EXP"]]
          .groupby("variant").agg(params=("params", "mean"),
                                  CC_temporal=("CC_temporal", "mean")).dropna())
    T8["M_params"] = (T8["params"] / 1e6).round(3)
    T8["CC_per_Mparam"] = (T8["CC_temporal"] / T8["M_params"]).round(2)
    T8 = T8.sort_values("CC_temporal", ascending=False)
    T8.index = [NICE.get(i, LAB6.get(i, i)) for i in T8.index]
    print("\n" + "=" * 96)
    print("TABLE 8  —  budget.  The baseline paper reports neither parameters nor FLOPs.")
    print("=" * 96)
    print(T8[["M_params", "CC_temporal", "CC_per_Mparam"]].to_string())
    T8.to_csv(WORK / "tables" / "table8_budget.csv")
MAJOR("03_tables_6_7_8")

---
# 6 · Figures

Ten figures, all written to `figures/` at 160 dpi and pushed. The house palette matches NB01–NB04
so the whole paper reads as one piece of work: **teal for radar (input), red for ECG (output)** —
the two accents mean something rather than decorating.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from crvs_metrics import bland_altman

S = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
     "grid": "#D3DADB", "amber": "#8A6212", "soft": "#9BB8BA"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": S["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 8.5,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "legend.fontsize": 7.5})
FIG = WORK / "figures"
def save(f, nm):
    f.savefig(FIG / nm); plt.close(f); print("  wrote", nm)

# --- F1 ours vs published --------------------------------------------------
if T3 is not None and len(T3):
    d = T3.dropna(subset=["CC_t_paper"])
    if len(d):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        for ax, ours, paper, ttl in [(axes[0], "CC_temporal", "CC_t_paper", "temporal correlation"),
                                     (axes[1], "CC_spectral", "CC_s_paper", "spectral correlation")]:
            xs = np.arange(len(d))
            ax.bar(xs - .2, d[ours], .4, color=S["radar"], edgecolor="white", label="our run")
            ax.bar(xs + .2, d[paper], .4, color=S["muted"], edgecolor="white", label="published")
            ax.set_xticks(xs); ax.set_xticklabels(d["variant"], rotation=20, ha="right", fontsize=7)
            ax.set_title(ttl + "  (RVA combined)")
            ax.legend(frameon=False)
        fig.tight_layout(); save(fig, "fig01_ours_vs_published.png")

# --- F2 ablation ladder ----------------------------------------------------
if lad:
    vals = [b[b["variant"] == v]["CC_temporal"].mean() for v in lad]
    cols = [S["ecg"] if v == "L9_full" else S["muted"] if v == "multireslinknet"
            else S["radar"] for v in lad]
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.barh(range(len(lad)), vals, color=cols, edgecolor="white")
    ax.set_yticks(range(len(lad))); ax.set_yticklabels([LAB6[v] for v in lad], fontsize=7.5)
    ax.invert_yaxis()
    ax.axvline(61.86, color=S["ink"], ls="--", lw=1.2, label="published MultiResLinkNet")
    for i, v in enumerate(vals):
        ax.text(v + .3, i, f"{v:.1f}", va="center", fontsize=7)
    ax.set_xlabel("temporal correlation (x100)")
    ax.set_title("Ablation ladder — what each contribution is worth", loc="left")
    ax.legend(frameon=False)
    save(fig, "fig02_ablation.png")

# --- F3/F4 Bland-Altman ----------------------------------------------------
if len(SD):
    for met, unit, fn in [("mean_hr_bpm", "bpm", "fig03_bland_altman_hr.png"),
                          ("rmssd_ms", "ms", "fig04_bland_altman_rmssd.png")]:
        picks = [v for v in ("multireslinknet", "L9_full") if v in set(SD["variant"])]
        if not picks or f"gt_{met}" not in SD.columns:
            continue
        fig, axes = plt.subplots(1, len(picks), figsize=(5.4 * len(picks), 3.6), squeeze=False)
        for ax, v in zip(axes[0], picks):
            d = SD[(SD["variant"] == v) & (SD["experiment"] == CFG["HEADLINE_EXP"])].dropna(
                subset=[f"gt_{met}", f"pr_{met}"])
            if not len(d):
                continue
            ba = bland_altman(d[f"pr_{met}"].to_numpy(), d[f"gt_{met}"].to_numpy())
            c = S["ecg"] if v == "L9_full" else S["muted"]
            ax.scatter(ba["mean"], ba["diff"], s=22, color=c, alpha=.75, edgecolor="white", lw=.5)
            ax.axhline(ba["bias"], color=S["ink"], lw=1.2)
            ax.axhline(ba["loa_hi"], color=S["ink"], ls="--", lw=1)
            ax.axhline(ba["loa_lo"], color=S["ink"], ls="--", lw=1)
            ax.axhline(0, color=S["grid"], lw=.8)
            ax.set_title(f"{NICE.get(v, LAB6.get(v, v))}\nbias {ba['bias']:+.2f}  "
                         f"LoA [{ba['loa_lo']:+.1f}, {ba['loa_hi']:+.1f}] {unit}", fontsize=8.5)
            ax.set_xlabel(f"mean of methods ({unit})")
            ax.set_ylabel(f"predicted - true ({unit})")
        fig.suptitle(f"Bland–Altman agreement — {met.replace('_',' ')}", y=1.02,
                     fontsize=10, fontweight="bold")
        fig.tight_layout(); save(fig, fn)

# --- F5 per-subject box plots ---------------------------------------------
if len(WD):
    picks = [v for v in ORDER if v in set(WD["variant"])][:6]
    d = WD[(WD["variant"].isin(picks)) & (WD["experiment"] == CFG["HEADLINE_EXP"])]
    if len(d):
        fig, ax = plt.subplots(figsize=(11, 3.8))
        data = [d[d["variant"] == v]["CC_temporal"].dropna().to_numpy() for v in picks]
        bp = ax.boxplot(data, labels=[NICE.get(v, LAB6.get(v, v)) for v in picks],
                        patch_artist=True, showfliers=False, widths=.6)
        for patch, v in zip(bp["boxes"], picks):
            patch.set_facecolor(S["ecg"] if v == "L9_full" else S["radar"])
            patch.set_alpha(.75); patch.set_edgecolor("white")
        for m in bp["medians"]:
            m.set_color(S["ink"]); m.set_linewidth(1.4)
        ax.set_ylabel("temporal correlation per window (x100)")
        ax.set_title("Distribution across held-out windows — means hide the tail", loc="left")
        ax.tick_params(axis="x", rotation=14, labelsize=7)
        save(fig, "fig05_distribution.png")

    dsub = d.groupby(["variant", "subject"])["CC_temporal"].mean().reset_index()
    if len(dsub):
        fig, ax = plt.subplots(figsize=(11, 3.6))
        for k, v in enumerate(picks):
            s = dsub[dsub["variant"] == v]
            ax.scatter(np.full(len(s), k) + np.random.uniform(-.14, .14, len(s)),
                       s["CC_temporal"], s=26,
                       color=S["ecg"] if v == "L9_full" else S["radar"],
                       alpha=.8, edgecolor="white", lw=.5)
        ax.set_xticks(range(len(picks)))
        ax.set_xticklabels([NICE.get(v, LAB6.get(v, v)) for v in picks], rotation=14, fontsize=7)
        ax.set_ylabel("per-subject mean CC_temporal (x100)")
        ax.set_title("Per-subject performance — one dot per held-out subject", loc="left")
        save(fig, "fig06_per_subject.png")

# --- F7 qualitative grid ---------------------------------------------------
sp = {}
for p in sorted((RUNS / "runs").glob("*/preds_sample.npz")):
    nm = p.parent.name
    for v in ORDER:
        if f"__{v}__" in nm and nm.startswith(CFG["HEADLINE_EXP"]) and v not in sp:
            sp[v] = p
picks = [v for v in ("multireslinknet", "L6_no_ssm", "L9_full") if v in sp]
if picks:
    z0 = np.load(sp[picks[0]])
    k = min(5, len(z0["y"]) - 1); tt = np.arange(1024) / 128.0
    fig, axes = plt.subplots(len(picks) + 1, 1, figsize=(11, 1.9 * (len(picks) + 1)), sharex=True)
    axes[0].plot(tt, z0["y"][k], lw=1.2, color=S["ink"])
    axes[0].set_ylabel("ground truth", rotation=0, ha="right", va="center", fontsize=8)
    for ax, v in zip(axes[1:], picks):
        z = np.load(sp[v]); kk = min(k, len(z["p"]) - 1)
        ax.plot(tt, z["p"][kk], lw=1.2,
                color=S["ecg"] if v == "L9_full" else S["muted"])
        ax.set_ylabel(NICE.get(v, LAB6.get(v, v)), rotation=0, ha="right", va="center", fontsize=7.5)
    for a in axes:
        a.tick_params(labelleft=False)
    axes[-1].set_xlabel("seconds")
    axes[0].set_title("Held-out reconstruction — the question is whether the QRS stays sharp",
                      loc="left")
    fig.tight_layout(); save(fig, "fig07_qualitative.png")

# --- F8 budget scatter -----------------------------------------------------
if "params" in R.columns and R["params"].notna().any():
    d = (R[R["experiment"] == CFG["HEADLINE_EXP"]]
         .groupby("variant").agg(p=("params", "mean"), c=("CC_temporal", "mean")).dropna())
    if len(d):
        fig, ax = plt.subplots(figsize=(7.6, 4.2))
        for v, r in d.iterrows():
            col = S["ecg"] if v == "L9_full" else S["muted"] if v in NICE else S["radar"]
            ax.scatter(r["p"] / 1e6, r["c"], s=110, color=col, edgecolor="white", lw=1, zorder=3)
            ax.annotate(NICE.get(v, LAB6.get(v, v)).split(".")[-1].strip(),
                        (r["p"] / 1e6, r["c"]), fontsize=6.8, xytext=(5, 4),
                        textcoords="offset points")
        ax.axhline(61.86, color=S["ink"], ls="--", lw=1, label="published MultiResLinkNet")
        ax.set_xlabel("parameters (millions)"); ax.set_ylabel("CC_temporal (x100)")
        ax.set_title("Accuracy against model size — smaller and better is the claim", loc="left")
        ax.legend(frameon=False)
        save(fig, "fig08_budget.png")
MAJOR("04_figures")

---
# 7 · Experiment E — robustness *(optional, needs GPU)*

The baseline never tests robustness. radarODE-MTL set the precedent that it matters, and a reviewer
will ask: what happens when the radar signal is noisier than a clinical recording room?

We add white Gaussian noise to the **input channels** at a range of SNRs and re-evaluate the saved
checkpoints. No retraining — this measures how gracefully each model degrades. Set
`CFG["RUN_ROBUSTNESS"] = False` to skip.

In [ ]:
if not CFG["RUN_ROBUSTNESS"]:
    print("robustness skipped (CFG['RUN_ROBUSTNESS'] = False)")
    ROB = pd.DataFrame()
else:
    import torch
    from crvs_data import WindowDataset, FS
    from crvs_models import build_baseline
    from crvs_cmnet import build_cmnet
    from crvs_metrics import seg_metrics
    from crvs_engine import pick_device, seed_all

    dev, ngpu, _ = pick_device()
    print("device:", dev, "| gpus:", ngpu)
    if dev.type != "cuda":
        print("  (CPU -- this will be slow; consider setting RUN_ROBUSTNESS=False)")

    DATA = SCRATCH / "corpus"
    snapshot_download(CFG["DATA_REPO"], repo_type="dataset", token=HF_TOKEN,
                      local_dir=str(DATA),
                      allow_patterns=["recordings/*.npy", "recordings/*.json",
                                      "recordings/*.npz",   # legacy corpus still works
                                      "windows.parquet", "norm_stats.json",
                                      "experiments.json"])
    Wn = pd.read_parquet(DATA / "windows.parquet")
    NORM = json.loads((DATA / "norm_stats.json").read_text())
    EXPINFO = json.loads((DATA / "experiments.json").read_text())
    EXPERIMENTS = EXPINFO["experiments"]
    REC_DIR = DATA / "recordings"

    def load_run(run_dir):
        s = json.loads((run_dir / "summary.json").read_text())
        v = norm_variant(s)
        spec = s.get("spec", {})
        ch = spec.get("channels") or s.get("channels") or ["dy"]
        if spec.get("kind") == "cmnet" or (v or "").startswith("L") and spec.get("kind") != "baseline":
            m = build_cmnet(in_ch=len(ch), base=32, levels=4, d_ssm=256, ssm_blocks=3,
                            bottleneck=spec.get("bottleneck", "ssm"),
                            use_wavelet=spec.get("wavelet", True),
                            multitask=spec.get("multitask", True),
                            use_film=spec.get("film", True))
        else:
            m = build_baseline(spec.get("model", s.get("model", "multireslinknet")),
                               in_ch=len(ch), out_ch=1, base=64, levels=4)
        sd = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)["model"]
        m.load_state_dict(sd, strict=False)
        return m.to(dev).eval(), ch, s, v

    rob = []
    seed_all(CFG["SEED"])
    for v in CFG["ROBUST_MODELS"]:
        cand = [p for p in (RUNS / "runs").glob(f"{CFG['HEADLINE_EXP']}__{v}__f*")
                if (p / "best.pt").exists() and (p / "summary.json").exists()]
        if not cand:
            print(f"  no checkpoint for {v} -- skipped"); continue
        rd = sorted(cand)[0]
        try:
            model, ch, s, vv = load_run(rd)
        except Exception as e:
            print(f"  could not load {rd.name}: {type(e).__name__}: {e}"); continue
        fold = s["fold"]
        sub = Wn[Wn["scenario_canon"].isin(EXPERIMENTS[CFG["HEADLINE_EXP"]])]
        te = sub[(sub["fold_group"] == fold % 5) & sub["no_overlap"]]
        te = te.iloc[:CFG["ROBUST_MAX_WINDOWS"]]
        norm = NORM[f"{CFG['HEADLINE_EXP']}|{fold}"]
        idx = [EXPINFO["channels"].index(c) for c in ch]
        sn = {"mean": [norm["mean"][i] for i in idx], "std": [norm["std"][i] for i in idx]}
        ds = WindowDataset(REC_DIR, te, sn, ch, augment=False)
        print(f"\n  {v}: {len(ds)} test windows, {len(ch)} channel(s)")
        for snr in CFG["SNR_DB"]:
            ccs = []
            with torch.no_grad():
                for i in range(0, len(ds), 32):
                    xb, yb = [], []
                    for j in range(i, min(i + 32, len(ds))):
                        x, y, _, _ = ds[j]; xb.append(x); yb.append(y)
                    X = torch.stack(xb).to(dev); Y = torch.stack(yb)
                    p_sig = X.pow(2).mean(dim=(1, 2), keepdim=True)
                    p_noise = p_sig / (10 ** (snr / 10.0))
                    X = X + torch.randn_like(X) * p_noise.sqrt()
                    out = model(X)["wave"].float().cpu().numpy()[:, 0]
                    Yn = Y.numpy()[:, 0]
                    for a, bb in zip(Yn, out):
                        ccs.append(seg_metrics(a, bb, FS)["CC_temporal"])
            rob.append({"variant": v, "snr_db": snr, "CC_temporal": float(np.mean(ccs)),
                        "n": len(ccs)})
            print(f"    SNR {snr:>4} dB   CC_t {np.mean(ccs):6.2f}")
        del model
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()

    ROB = pd.DataFrame(rob)
    if len(ROB):
        ROB.to_csv(WORK / "tables" / "table9_robustness.csv", index=False)
        fig, ax = plt.subplots(figsize=(7.4, 3.8))
        for v in ROB["variant"].unique():
            d = ROB[ROB["variant"] == v].sort_values("snr_db", ascending=False)
            ax.plot(d["snr_db"], d["CC_temporal"], "o-", lw=1.6, ms=5,
                    color=S["ecg"] if v == "L9_full" else S["muted"],
                    label=NICE.get(v, LAB6.get(v, v)))
        ax.invert_xaxis()
        ax.set_xlabel("input SNR (dB) — noisier to the right")
        ax.set_ylabel("CC_temporal (x100)")
        ax.set_title("Experiment E — graceful degradation under input noise", loc="left")
        ax.legend(frameon=False)
        save(fig, "fig09_robustness.png")
MAJOR("05_robustness")

---
# 8 · Manuscript-ready summary and final push

A single `RESULTS.md` collecting every table, so drafting Section 6 of the paper is transcription
rather than archaeology.

In [ ]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
L = []
A = L.append
A(f"# CardioMamba-Net — results\n\nGenerated {now} from `{CFG['MODEL_REPO']}`.\n")
A(f"- runs analysed: **{len(R)}**")
A(f"- experiments: {sorted(R['experiment'].unique())}")
A(f"- variants: {sorted(R['variant'].unique())}\n")

if T3 is not None and len(T3):
    A("## Table 3 — RVA combined (headline)\n")
    A(T3.round(5).to_markdown(index=False)); A("")
if T2 is not None and len(T2):
    A("## Table 2 — per scenario\n"); A(T2.round(5).to_markdown(index=False)); A("")
if TC is not None and len(TC):
    A("## Table 3b — all five scenarios (new)\n"); A(TC.round(5).to_markdown(index=False)); A("")
try:
    A("## Table 4 — R-peak detection\n"); A(T4.to_markdown()); A("")
    A("## Table 5 — HR and HRV (real ms)\n"); A(T5.to_markdown(index=False)); A("")
except Exception:
    pass
try:
    A("## Table 6 — ablation ladder\n"); A(T6.round(5).to_markdown()); A("")
    A("## Table 7 — Wilcoxon signed-rank, Holm-corrected\n"); A(T7.to_markdown(index=False)); A("")
    A("## Table 8 — budget\n"); A(T8[["M_params","CC_temporal","CC_per_Mparam"]].to_markdown()); A("")
except Exception:
    pass
if CFG["RUN_ROBUSTNESS"] and len(ROB):
    A("## Table 9 — robustness\n"); A(ROB.to_markdown(index=False)); A("")

A("## Figures\n")
for p in sorted(FIG.glob("*.png")):
    A(f"- `figures/{p.name}`")
A("\n## Reading notes for the manuscript\n")
A("- Our splits are strictly subject-wise with non-overlapping test windows. The baseline's "
  "Table 1 counts carry the 50 % overlap and are split 80/20, which permits overlapping windows "
  "across train and test. Baseline rows landing below their published values is the expected "
  "consequence of removing that, not a weaker implementation.")
A("- Correlations are reported x100 throughout, matching the baseline's tables.")
A("- mu_RR is in genuine milliseconds. The baseline's Table 5 reports 126 ms alongside 62 bpm, "
  "which is arithmetically impossible; 126 samples at 128 Hz is 0.98 s.")
A("- Significance is Wilcoxon signed-rank across folds with Holm correction. With ten ablation "
  "rungs there are 45 pairwise tests, so uncorrected p-values would be misleading.")
(WORK / "RESULTS.md").write_text("\n".join(L))

sizes = {str(p.relative_to(WORK)): p.stat().st_size for p in WORK.rglob("*") if p.is_file()}
ok = sync.flush(final=True, msg=f"{CFG['RUN_ID']} — evaluation complete, {len(R)} runs")
print("\n" + "=" * 76)
print("  EVALUATION COMPLETE" if ok else "  COMPLETE (final push had a problem)")
print("=" * 76)
print(f"  repo    : {sync.url}")
print(f"  runs    : {len(R)}")
print(f"  tables  : {len(list((WORK/'tables').glob('*.csv')))}")
print(f"  figures : {len(list(FIG.glob('*.png')))}")
print(f"  payload : {sum(sizes.values())/2**20:.1f} MB")
print("=" * 76)
print("\n  RESULTS.md holds every table in markdown, ready to paste into the manuscript.")

---
# 9 · Troubleshooting

**`No runs found`** — NB03 and NB04 push to `CFG["MODEL_REPO"]`. Check the repo name matches and
that at least one run finished.

**`TABLE 7 skipped: only 1 fold`** — expected while `QUICK = True`. Wilcoxon needs at least three
paired observations; run NB03/NB04 with `QUICK = False` so all five folds complete.

**Bland–Altman plots empty** — the per-subject metrics come from `metrics_subjects.parquet`, which
is only written when a test split has at least two windows per subject. Re-run with the full folds.

**Robustness section slow or out of memory** — lower `CFG["ROBUST_MAX_WINDOWS"]`, or set
`CFG["RUN_ROBUSTNESS"] = False` and run it in its own session.

**A checkpoint fails to load** — architecture config drifted between training and evaluation. The
loader uses `strict=False` so partial mismatches warn rather than crash, but the numbers would then
be meaningless. If you changed `CFG["BASE"]` or `D_SSM` in NB04 after training, set them back.